In [19]:
import numpy as np 
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score

In [3]:
df = pd.read_csv("../data/Titanic-Dataset.csv")

In [4]:
X = df.drop(columns = ["Survived"])
y = df["Survived"]

In [5]:
numerical_feature = X.select_dtypes(include = 'number').columns
categorical_feature = X.select_dtypes(include = 'object').columns

In [6]:
numeric_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy = 'median')),
    ('scaling',StandardScaler())
])

In [7]:
categorical_pipeline = Pipeline([
    ('imputer',SimpleImputer(strategy = 'most_frequent')),
    ('encoder',OneHotEncoder(handle_unknown = 'ignore'))
    
])

In [8]:
transformer = ColumnTransformer([
    ('numeric_pipe',numeric_pipeline,numerical_feature),
    ('categoric_pipe',categorical_pipeline,categorical_feature)
])


In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [10]:
from sklearn.linear_model import LogisticRegression
Logi_pipeline = Pipeline([
    ('preprosser',transformer),
    ('model',LogisticRegression(max_iter=1000))
])



In [11]:
param_grid = {
    'model__C': [0.01, 0.1, 1, 10, 100],
    'model__solver': ['liblinear', 'lbfgs']
}

In [12]:
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    Logi_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)


In [13]:
grid_search.fit(X_train,y_train)

,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'model__C': [0.01, 0.1, ...], 'model__solver': ['liblinear', 'lbfgs']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('numeric_pipe', ...), ('categoric_pipe', ...)]"


In [14]:
print(grid_search.best_params_)
print(grid_search.best_score_)

{'model__C': 100, 'model__solver': 'liblinear'}
0.8146557667684429


In [15]:
from sklearn.metrics import accuracy_score, f1_score

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.8100558659217877
F1: 0.746268656716418


In [16]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, y_pred))

[[95 15]
 [19 50]]


In [37]:
cv_score = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=5,
    scoring = "accuracy"
)

In [38]:
f1_score = cross_val_score(
    best_model,
    X_train,
    y_train,
    cv=5,
    scoring = "f1"
)

In [39]:
print(cv_score)
print(cv_score.mean())

[0.81118881 0.78321678 0.84507042 0.81690141 0.81690141]
0.8146557667684429


In [40]:
print(cv_score.std())
print(f1_score.std())

0.01967519538044744
0.037201136517939604


In [41]:
from sklearn.metrics import f1_score

y_pred = best_model.predict(X_test)

test_f1 = f1_score(y_test, y_pred)

print(test_f1)

0.746268656716418
